In [1]:
# %load_ext autoreload
import seaborn as sns
import numpy as np
import pandas as pd
import pathlib
from matplotlib import pyplot as plt
import json
from bs4 import BeautifulSoup
import re
import importlib
import fnmodules
import datetime

In [101]:
source_data_path = pathlib.Path.cwd().parent / "Data Repo" / "SSAT" / "SSAT_DATA_2022-06-08_1235.csv"
source_dc_path = pathlib.Path.cwd().parent / "Data Dictionaries" / "Processed DataCompanion" / "DataCompanion_SSATRAW_20220609_115434.xlsx"
dc = pd.read_excel(source_dc_path,sheet_name=None)

dtypemap = {"KEY"       :"string",
            "META"      :"string",
            "INSTANCE"  :"UInt16",
            "INT"       :"Int64",
            # "DATE"      :"datetime64[ns]",
            "LETTERS"   :"string",
            "TEXT"      :"string",
            # "CATEGORIC" :"",
            "DOUBLE"    :"float64",
            # "DATETIME"  :"datetime64[ns]",
            # "TIME"      :"timedelta64[ns]",
            }
vardtypes = dc.get("Vars").set_index("VARNAME")["DATATYPE"].map(dtypemap).dropna().to_dict()
def f(x):
    return pd.api.types.CategoricalDtype(list(json.loads(x).keys()),ordered=True)
categvars = dc.get("Vars")[dc.get("Vars")["DATATYPE"] == "CATEGORIC"].set_index("VARNAME")["CATEGORICCODES"].to_dict()
categvars = {y: f(x) for y,x in categvars.items()}
vardtypes.update(categvars)

datevars = dc.get("Vars")[(dc.get("Vars")["DATATYPE"] == "DATE") | (dc.get("Vars")["DATATYPE"] == "DATETIME")]["VARNAME"].to_list()
timevars = dc.get("Vars")[(dc.get("Vars")["DATATYPE"] == "TIME")]["VARNAME"].to_list()

# print(timevars)
df = pd.read_csv(source_data_path,dtype=vardtypes,parse_dates=datevars)
df[timevars] = df[timevars].apply(lambda x: pd.to_timedelta(x + ":00"))
print(df.info(verbose=True))
for col in df:
    print(df[col].describe(datetime_is_numeric=True))
    print()
# df["study_id"].head(15)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1833 entries, 0 to 1832
Data columns (total 374 columns):
 #    Column                                                   Dtype          
---   ------                                                   -----          
 0    record_id                                                string         
 1    redcap_event_name                                        string         
 2    redcap_repeat_instrument                                 string         
 3    redcap_repeat_instance                                   UInt16         
 4    screen_id                                                Int64          
 5    screen_date                                              datetime64[ns] 
 6    screen_initials                                          string         
 7    screen_initials_text                                     string         
 8    screen_d_gender                                          category       
 9    screen_d_dob     

In [3]:
importlib.reload(fnmodules)
transform_from = pathlib.Path.cwd().parent / "Data Transformation" / "DataTransform_SSAT_try1.xlsx"
df_trans = pd.read_excel(transform_from,sheet_name=None)
dataclass = fnmodules.toolbox()
for _, action_item in df_trans.get("ActionList").iterrows():
    if action_item["IGNORE"] != True:
        method = getattr(dataclass, action_item["ACTION"])
        print(f"[STARTING] {action_item['ACTION']}")
        method(target = {1:action_item["TARGET"],2:action_item["TARGET2"]},saveto = {1:action_item["SAVETO"],2:action_item["SAVETO2"]} \
              ,params = {1:action_item["PARAM1"],2:action_item["PARAM2"],3:action_item["PARAM3"],4:action_item["PARAM4"]})
        print(f"[DONE] {action_item['ACTION']}")
        print()
dataclass.v.keys()
# a = list(dataclass.v.get('dfraw').keys())
# b = list(dataclass.v.get('dcraw').get("Vars")["VARNAME"])
# c = list(dataclass.v.get('dcraw').get("Vars")["DATATYPE"])

# print(len([ai for ai in a if ai in b]))
# print([ai for ai in a if ai not in b])
# print(len([bi for bi in b if bi in a]))
# print([bi for bi in b if bi not in a])
# print([ci for bi, ci in zip(b,c) if bi not in a])
# dataclass.v.get('dcraw').get("Metadata")


[STARTING] loadfile


c:\Users\eocru-lp010\Documents\W\EOCRU\Projects\Data Automation\datatools_oucru\fnmodules.py:40: DtypeWarning: Columns (161,163,220,232,236,240,241) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(utils.fetch_path("Data",mdata.at["LocationRawData","value"]),dtype=vardtypes,parse_dates=datevars)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3202 entries, 0 to 3201
Data columns (total 386 columns):
 #    Column                                                   Dtype          
---   ------                                                   -----          
 0    record_id                                                string         
 1    redcap_event_name                                        string         
 2    redcap_repeat_instrument                                 string         
 3    redcap_repeat_instance                                   UInt16         
 4    screen_id                                                Int64          
 5    screen_date                                              datetime64[ns] 
 6    screen_initials                                          string         
 7    screen_initials_text                                     object         
 8    screen_d_gender                                          category       
 9    screen_d_dob     

dict_keys(['dcraw', 'dfraw', 'dcraw2', 'dfraw2'])

In [33]:
dataclass.v.get("dfraw2").keys()


dict_keys(['Non-repeating'])

In [106]:
dataclass.v.get("dfraw2").get("Non-repeating").info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 376 columns):
 #    Column                           Dtype         
---   ------                           -----         
 0    record_id                        string        
 1    invite_id                        string        
 2    age                              string        
 3    eocru_initials                   string        
 4    pmi_complete                     category      
 5    eocru_eligibility___1            category      
 6    eocru_eligibility___2            category      
 7    eocru_eligibility___3            category      
 8    eocru_vac_path                   category      
 9    eocru_consent1                   category      
 10   eocru_consent2                   category      
 11   eocru_consent3                   category      
 12   date_of_enrolment                datetime64[ns]
 13   gender                           category      
 14   ethnicity___1           

In [407]:
dx = dataclass.v.get("dfraw2").get("Non-repeating").copy()
dx2 = dataclass.v.get("dfraw2").get("Patient List").copy()
dc = dataclass.v.get('dcraw2').get("Vars").copy()
dx_past = None

dx = dx.set_index("record_id")
dx["redcap_data_access_group"] = dx2[("","redcap_data_access_group")]
for cat in dx.select_dtypes(include=["category"]):
    dx[cat] = dx[cat].cat.rename_categories(json.loads(dc.set_index("VARNAME").loc[cat,"CATEGORICCODES"]))

In [441]:
for idy in dy.keys():
    for cat in dy[idy].select_dtypes(include=["category"]):
        dy[idy][cat] = dy[idy][cat].cat.rename_categories(json.loads(dd.set_index("VARNAME").loc[cat,"CATEGORICCODES"]))

In [631]:


report = {}
table_new = {}

def add_total_col(df,sort=True):
    x = df.assign(Total = lambda df: df.sum(axis=1))
    if sort:
        return x.sort_values("Total",ascending=False)
    else:
        return x

#===================================================================================
# Segment 1 ========================================================================
#===================================================================================
s1 = {}
s1y = {}
    #Number of patients enrolled
s1["Total Enrolled"] = dx.value_counts("redcap_data_access_group")
s1y["Total Enrolled"] = dy.get("Registration").value_counts("pmi_dag_default")
    #Number of enrolment in last 30 days
n_days = 30
s1["Enrol last 30 days"] = dx[(dx["date_of_enrolment"] >= datetime.datetime.now() - pd.Timedelta(n_days,unit="d")) & (dx["date_of_enrolment"] <= datetime.datetime.now())].value_counts("redcap_data_access_group")
s1y["Enrol last 30 days"] = dy.get("Registration")[(dy.get("Registration")["eli_date_enrol"] >= datetime.datetime.now() - pd.Timedelta(n_days,unit="d")) & (dy.get("Registration")["eli_date_enrol"] <= datetime.datetime.now())].value_counts("pmi_dag_default")
    #ERROR: Enrolment date is after TODAY
dx[(dx["date_of_enrolment"] > datetime.datetime.now())]["date_of_enrolment"]
    #Patients with 2 Vaccines
s1["Participant with V1 V2"] = dx[(dx["pitch_had_covid_vac"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes')].value_counts("redcap_data_access_group")
s1y["Participant with V1 V2"] = dy.get("Registration")[(dy.get("Vaccine #1")["vac_check"] == 'Yes') & (dy.get("Vaccine #2")["vac_check"] == 'Yes')].value_counts("pmi_dag_default")
    #Community cohort with completed followups (V1 V2 V2M1 V2M3)
s1["Total Community cohort"] = dx[(dx["eocru_vac_path"] == 'Scheduled to be receiving first and second vaccine dose, at same site (community cohort)')].value_counts("redcap_data_access_group")
s1["Community with V1 V2"] = dx[(dx["eocru_vac_path"] == 'Scheduled to be receiving first and second vaccine dose, at same site (community cohort)') & (dx["pitch_had_covid_vac"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes')].value_counts("redcap_data_access_group")
s1["Community with V1 V2 V2M1 V2M3"] = dx[(dx["eocru_vac_path"] == 'Scheduled to be receiving first and second vaccine dose, at same site (community cohort)') & (dx["pitch_had_covid_vac"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes') & (dx["pitch_visit_date_v2d28"].notna()) & (dx["pitch_visit_date_v2m3"].notna())].value_counts("redcap_data_access_group")

s1y["Total Community cohort"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"].isin(['Scheduled to be receiving first and second vaccine dose, at same site (Community Cohort)','Scheduled to be receiving booster vaccine dose (Community cohort)']))].value_counts("pmi_dag_default")
s1y["Community with V1 V2"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"].isin(['Scheduled to be receiving first and second vaccine dose, at same site (Community Cohort)','Scheduled to be receiving booster vaccine dose (Community cohort)'])) 
            & (dy.get("Vaccine #1")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #2")["vac_check"] == 'Yes')].value_counts("pmi_dag_default")
s1y["Community with V1 V2 V2M1 V2M3"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"].isin(['Scheduled to be receiving first and second vaccine dose, at same site (Community Cohort)','Scheduled to be receiving booster vaccine dose (Community cohort)'])) 
            & (dy.get("Vaccine #1")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #2")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #2")["m1_date"].notna()) 
            & (dy.get("Vaccine #2")["m3_date"].notna())].value_counts("pmi_dag_default")
    # HCW cohort with completed followups (V3 V3M1 V3M3 V3M6)
s1["Total HCW cohort"] = dx[(dx["eocru_vac_path"] == 'Vaccinated hospital staff at risk of exposure to SARS-CoV-2 (health care worker cohort)')].value_counts("redcap_data_access_group")
s1["HCW with V1 V2 V3"] = dx[(dx["eocru_vac_path"] == 'Vaccinated hospital staff at risk of exposure to SARS-CoV-2 (health care worker cohort)') & (dx["pitch_had_covid_vac"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes') & (dx["pitch_had_covid_vac3"] == 'Yes')].value_counts("redcap_data_access_group")
s1["HCW with V1 V2 V3 V3M1 V3M3 V3M6"] = dx[(dx["eocru_vac_path"] == 'Vaccinated hospital staff at risk of exposure to SARS-CoV-2 (health care worker cohort)') & (dx["pitch_had_covid_vac"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes') & (dx["pitch_had_covid_vac2"] == 'Yes') & (dx["pitch_had_covid_vac3"] == 'Yes') & (dx["pitch_visit_date_v3d28"].notna()) & (dx["pitch_visit_date_v3m3"].notna()) & (dx["pitch_visit_date_v3m6"].notna())].value_counts("redcap_data_access_group")

s1y["Total HCW cohort"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"] == "Vaccinated hospital staff at risk of exposure to SARS-CoV-2, scheduled to receive booster dose (Healthcare worker)")].value_counts("pmi_dag_default")
s1y["HCW with V1 V2 V3"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"] == "Vaccinated hospital staff at risk of exposure to SARS-CoV-2, scheduled to receive booster dose (Healthcare worker)") 
            & (dy.get("Vaccine #1")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #2")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #3")["vac_check"] == 'Yes')].value_counts("pmi_dag_default")
s1y["HCW with V1 V2 V3 V3M1 V3M3 V3M6"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"] == "Vaccinated hospital staff at risk of exposure to SARS-CoV-2, scheduled to receive booster dose (Healthcare worker)") 
            & (dy.get("Vaccine #1")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #2")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #3")["vac_check"] == 'Yes') 
            & (dy.get("Vaccine #3")["m1_date"].notna()) 
            & (dy.get("Vaccine #3")["m3_date"].notna()) 
            & (dy.get("Vaccine #3")["m6_date"].notna())].value_counts("pmi_dag_default")
s1 = add_total_col(pd.DataFrame(s1).T.fillna(0),sort=False)
s1y = add_total_col(pd.DataFrame(s1y).T.fillna(0),sort=False)
report["Enrollment Status"] = s1
report["Enrollment Status (OUCRU)"] = s1y



C:\Users\eocru-lp010\AppData\Local\Temp\ipykernel_14064\1966538997.py:27: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s1y["Participant with V1 V2"] = dy.get("Registration")[(dy.get("Vaccine #1")["vac_check"] == 'Yes') & (dy.get("Vaccine #2")["vac_check"] == 'Yes')].value_counts("pmi_dag_default")
C:\Users\eocru-lp010\AppData\Local\Temp\ipykernel_14064\1966538997.py:34: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s1y["Community with V1 V2"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"].isin(['Scheduled to be receiving first and second vaccine dose, at same site (Community Cohort)','Scheduled to be receiving booster vaccine dose (Community cohort)']))
C:\Users\eocru-lp010\AppData\Local\Temp\ipykernel_14064\1966538997.py:37: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  s1y["Community with V1 V2 V2M1 V2M3"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"].

In [642]:
#===================================================================================
# Segment 4 ========================================================================
#===================================================================================

serumvars = ['SV1','SV2','+SV2M1','+SV2M3','+SV2M6','+SV2M12','SV3','+SV3M1','+SV3M3','+SV3M6','+SV3M12']
dx['SV1'] = (dx["eocru_serum_prev1"] == 'Yes') & (dx["eocru_pbmc_prev1"] == 'Yes')
dx['SV2'] = (dx["SV1"] == 1) & (dx["eocru_serum_prev2"] == 'Yes') & ((dx["eocru_pbmc_prev1"] == 'No') | (dx["eocru_pbmc_prev2"] == 'Yes'))
dx['+SV2M1'] = ((dx["SV2"] == 1) 
            & (dx["pitch_serum_visit_v2d28"] == 'Yes') 
            & ((dx["eocru_pbmc_prev1"] == 'No') | (dx["pitch_pbmc_visit_v2d28"] == 'Yes')))
dx['+SV2M3'] = ((dx["+SV2M1"] == 1) 
            & (dx["pitch_serum_visit_v2m3"] == 'Yes') 
            & ((dx["eocru_pbmc_prev1"] == 'No') | (dx["pitch_pbmc_visit_v2m3"] == 'Yes')))
dx['+SV2M6'] = ((dx["+SV2M3"] == 1) 
            & (dx["pitch_serum_visit_v2m6"] == 'Yes') 
            & ((dx["eocru_pbmc_prev1"] == 'No') | (dx["pitch_pbmc_visit_v2m6"] == 'Yes')))
dx['+SV2M12'] = ((dx["+SV2M6"] == 1) 
            & (dx["pitch_serum_visit_v2m12"] == 'Yes') 
            & ((dx["eocru_pbmc_prev1"] == 'No') | (dx["pitch_pbmc_visit_v2m12"] == 'Yes')))
            
dx['SV3'] = (dx["eocru_pbmc_prev3"] == 'Yes')  & (dx["eocru_serum_prev3"] == 'Yes')
dx['+SV3M1'] = ((dx["SV3"] == 1) 
            & (dx["eocru_serum_v3d28"] == 'Yes') 
            & ((dx["eocru_pbmc_prev3"] == 'No') | (dx["eocru_pbmc_v3d28"] == 'Yes')))
dx['+SV3M3'] = ((dx["+SV3M1"] == 1) 
            & (dx["eocru_serum_v3m3"] == 'Yes') 
            & ((dx["eocru_pbmc_prev3"] == 'No') | (dx["eocru_pbmc_v3m3"] == 'Yes')))
dx['+SV3M6'] = ((dx["+SV3M3"] == 1) 
            & (dx["eocru_serum_v3m6"] == 'Yes') 
            & ((dx["eocru_pbmc_prev3"] == 'No') | (dx["eocru_pbmc_v3m6"] == 'Yes')))
dx['+SV3M12'] = ((dx["+SV3M6"] == 1) 
            & (dx["eocru_serum_v3m12"] == 'Yes') 
            & ((dx["eocru_pbmc_prev3"] == 'No') | (dx["eocru_pbmc_v3m12"] == 'Yes')))

s4 = dx.groupby(["eocru_vac_path"])[serumvars].sum()


dy['Registration']['SV1'] = (dy.get("Vaccine #1")["prevac_pbmc"] == "Yes") & (dy.get("Vaccine #1")["prevac_serum"] == "Yes")
dy['Registration']['SV2'] = ((dy.get("Registration")["SV1"] == 1) & ((dy.get("Vaccine #1")["prevac_pbmc"] == "No") | (dy.get("Vaccine #2")["prevac_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #2")["prevac_serum"] == "Yes"))
dy['Registration']['+SV2M1'] = ((dy.get("Registration")["SV2"] == 1) 
                        & ((dy.get("Vaccine #1")["prevac_pbmc"] == "No") | (dy.get("Vaccine #2")["m1_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #2")["m1_serum"] == "Yes"))
dy['Registration']['+SV2M3'] = ((dy.get("Registration")["+SV2M1"] == 1) 
                        & ((dy.get("Vaccine #1")["prevac_pbmc"] == "No") | (dy.get("Vaccine #2")["m3_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #2")["m3_serum"] == "Yes"))
dy['Registration']['+SV2M6'] = ((dy.get("Registration")["+SV2M3"] == 1) 
                        & ((dy.get("Vaccine #1")["prevac_pbmc"] == "No") | (dy.get("Vaccine #2")["m6_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #2")["m6_serum"] == "Yes"))
dy['Registration']['+SV2M12'] = ((dy.get("Registration")["+SV2M6"] == 1) 
                        & ((dy.get("Vaccine #1")["prevac_pbmc"] == "No") | (dy.get("Vaccine #2")["m12_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #2")["m12_serum"] == "Yes"))
                        
dy['Registration']['SV3'] = ((dy.get("Vaccine #3")["prevac_pbmc"] == "Yes") 
                        & (dy.get("Vaccine #3")["prevac_serum"] == "Yes"))
dy['Registration']['+SV3M1'] = ((dy.get("Registration")["SV3"] == 1) 
                        & ((dy.get("Vaccine #3")["prevac_pbmc"] == "No") | (dy.get("Vaccine #3")["m1_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #3")["m1_serum"] == "Yes"))
dy['Registration']['+SV3M3'] = ((dy.get("Registration")["+SV3M1"] == 1) 
                        & ((dy.get("Vaccine #3")["prevac_pbmc"] == "No") | (dy.get("Vaccine #3")["m3_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #3")["m3_serum"] == "Yes"))
dy['Registration']['+SV3M6'] = ((dy.get("Registration")["+SV3M3"] == 1) 
                        & ((dy.get("Vaccine #3")["prevac_pbmc"] == "No") | (dy.get("Vaccine #3")["m6_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #3")["m6_serum"] == "Yes"))
dy['Registration']['+SV3M12'] = ((dy.get("Registration")["+SV3M6"] == 1) 
                        & ((dy.get("Vaccine #3")["prevac_pbmc"] == "No") | (dy.get("Vaccine #3")["m12_pbmc"] == "Yes")) 
                        & (dy.get("Vaccine #3")["m12_serum"] == "Yes"))

s4y = dy.get("Registration").groupby(["eli_vaccpath"])[serumvars].sum()
# s4y["HCW"] = dy.get("Registration")[(dy.get("Registration")["eli_vaccpath"] == "Vaccinated hospital staff at risk of exposure to SARS-CoV-2, scheduled to receive booster dose (Healthcare worker)") 
#             & (dy.get("Vaccine #1")["prevac_serum"] == 'Yes') 
#             & (dy.get("Vaccine #1")["prevac_pbmc"] == 'Yes') 
#             & (dy.get("Vaccine #2")["prevac_serum"] == 'Yes') 
#             & (dy.get("Vaccine #2")["prevac_pbmc"] == 'Yes')].value_counts("pmi_dag_default")
# s4["HCW"]


report["Completed Samples"] = s4
report["Completed Samples (OUCRU)"] = s4y
            

Series([], dtype: bool)

In [9]:
import pandas as pd
s = pd.Series(pd.Categorical(["a", "b", "b", "a", "a", "d"]))
s.map({"a":"good","b":"very good","d":"good"}).astype("category")
# s

0         good
1    very good
2    very good
3         good
4         good
5         good
dtype: category
Categories (2, object): ['good', 'very good']

In [633]:
#===================================================================================
# Segment 2 ========================================================================
#===================================================================================
s2a = {}
s2ay = {}
    #Sort order
vacc_prevalence_sort_ord = (dx["oucru_vac_manufa_first"].value_counts() + dx["oucru_vac2_manufa"].value_counts() + dx["eocru_vac3_manufa"].value_counts()).sort_values(ascending=False).index.to_list()
vacc_prevalence_sort_ord_y = (dy.get("Vaccine #1")["vac_manufacturer"].value_counts() + dy.get("Vaccine #2")["vac_manufacturer"].value_counts() + dy.get("Vaccine #3")["vac_manufacturer"].value_counts()).sort_values(ascending=False).index.to_list()
    #Report of individual vaccine number prevalency
s2a["V1 Vaccine Types"] = dx["oucru_vac_manufa_first"].value_counts()[vacc_prevalence_sort_ord]
s2a["V2 Vaccine Types"] = dx["oucru_vac2_manufa"].value_counts()[vacc_prevalence_sort_ord]
s2a["V3 Vaccine Types"] = dx["eocru_vac3_manufa"].value_counts()[vacc_prevalence_sort_ord]
s2a = add_total_col(pd.DataFrame(s2a).T)
s2a
s2ay["V1 Vaccine Types"] = dy.get("Vaccine #1")["vac_manufacturer"].value_counts()[vacc_prevalence_sort_ord]
s2ay["V2 Vaccine Types"] = dy.get("Vaccine #2")["vac_manufacturer"].value_counts()[vacc_prevalence_sort_ord]
s2ay["V3 Vaccine Types"] = dy.get("Vaccine #3")["vac_manufacturer"].value_counts()[vacc_prevalence_sort_ord]
s2ay = add_total_col(pd.DataFrame(s2ay).T)
s2ay
    #Report of Vaccine pairs, ordered (1-2-3) combinations
dx['unordered_vacc_manufa'] = dx[["oucru_vac_manufa_first","oucru_vac2_manufa","eocru_vac3_manufa"]].astype("string").fillna("x").agg(lambda x: f"{x['oucru_vac_manufa_first']} --- {x['oucru_vac2_manufa']} --- {x['eocru_vac3_manufa']}", axis=1)
s2b = add_total_col(dx.groupby("redcap_data_access_group")['unordered_vacc_manufa'].value_counts().unstack().T.fillna(0))
dy['Registration']['unordered_vacc_manufa'] = pd.DataFrame({ "V1":dy.get("Vaccine #1")["vac_manufacturer"]
    ,"V2":dy.get("Vaccine #2")["vac_manufacturer"]
    ,"V3":dy.get("Vaccine #3")["vac_manufacturer"]}).astype("string").fillna("x").agg(lambda x: f"{x['V1']} --- {x['V2']} --- {x['V3']}", axis=1)
s2by = add_total_col(dy.get("Registration").groupby("pmi_dag_default")['unordered_vacc_manufa'].value_counts().unstack().T.fillna(0))
s2by
    #Report of Vaccine pairs, unordered combinations
dx['sorted_vacc_manufa'] = [' --- '.join(sorted([str(x) , str(y) , str(z)])) for x, y, z in zip(
            dx['oucru_vac_manufa_first'].astype("string").fillna("x")
            , dx['oucru_vac2_manufa'].astype("string").fillna("x")
            , dx['eocru_vac3_manufa'].astype("string").fillna("x"))]
s2c = add_total_col(dx.groupby("redcap_data_access_group")['sorted_vacc_manufa'].value_counts().unstack().T.fillna(0))
s2c

dy['Registration']['sorted_vacc_manufa'] = pd.Series({k:' --- '.join(sorted([v["V1"] , v["V2"] , v["V3"]])) for k, v in 
    pd.DataFrame({ "V1":dy.get("Vaccine #1")["vac_manufacturer"]
    ,"V2":dy.get("Vaccine #2")["vac_manufacturer"]
    ,"V3":dy.get("Vaccine #3")["vac_manufacturer"]}).astype('string').fillna("x").to_dict("index").items()})
s2cy = add_total_col(dy.get("Registration").groupby("pmi_dag_default")['unordered_vacc_manufa'].value_counts().unstack().T.fillna(0))
s2cy
report["Vac Type - Indiv"] = s2a
report["Vac Type - Comp Order"] = s2b
report["Vac Type - Comp Any Order"] = s2c
report["Vac Type - Indiv (OUCRU)"] = s2ay
report["Vac Type - Comp Order (OUCRU)"] = s2by
report["Vac Type - Comp Any Order (OUCRU)"] = s2cy
# s2c["Total"] = s2c.sum(axis=1)
# s2c = s2c.sort_values()


In [643]:
#===================================================================================
# Segment 3 ========================================================================
#===================================================================================
dx[dx["eocru_pbmc_prev1"] == 'Yes'].shape[0]
dx[dx["eocru_pbmc_prev2"] == 'Yes'].shape[0]
dx[dx["pitch_pbmc_visit_v2d28"] == 'Yes'].shape[0]
dx[dx["pitch_pbmc_visit_v2m3"] == 'Yes'].shape[0]
dx[dx["pitch_pbmc_visit_v2m6"] == 'Yes'].shape[0]
dx[dx["pitch_pbmc_visit_v2m12"] == 'Yes'].shape[0]
dx[dx["eocru_pbmc_prev3"] == 'Yes'].shape[0]
dx[dx["eocru_pbmc_v3d28"] == 'Yes'].shape[0]
dx[dx["eocru_pbmc_v3m3"] == 'Yes'].shape[0]
dx[dx["eocru_pbmc_v3m6"] == 'Yes'].shape[0]
dx[dx["eocru_pbmc_v3m12"] == 'Yes'].shape[0]

cols = ["Total","gender","eocru_vac_path","illness_count"]
# dx["illnesses_number"].dtype
dx["Total"] = ""
dx["illness_count"] = pd.Series(dx["illnesses_number"]).cat.add_categories(["0"])
dx.loc[dx["covid_like_illness"] == "No","illness_count"] = "0"
s3 = dx[dx["eocru_pbmc_prev1"] == 'Yes'][cols].apply(lambda x: x.value_counts()).T.stack().rename("Count")
report["PBMC+ demographic"] = s3

colsy = ["Total","eli_gender","eli_vaccpath","illness_count"]
dy["Registration"]["Total"] = ""
dy["Registration"]["illness_count"]  = pd.Series(dy.get("Registration")["covhist_count"]).cat.add_categories(["0"])
dy["Registration"].loc[dy.get("Registration")["eli_covhist"] == "No","illness_count"] = "0"
s3y = dy.get("Registration")[dy.get("Vaccine #1")["prevac_pbmc"] == 'Yes'][colsy].apply(lambda x: x.value_counts()).T.stack().rename("Count")
report["PBMC+ demog (OUCRU)"] = s3y
s3

Total                                                                                                       282.0
gender          Female                                                                                      128.0
                Male                                                                                        154.0
                Other                                                                                         0.0
                Prefer not to say                                                                             0.0
eocru_vac_path  Scheduled to be receiving first and second vaccine dose, at same site (community cohort)    279.0
                Vaccinated hospital staff at risk of exposure to SARS-CoV-2 (health care worker cohort)       3.0
illness_count   0                                                                                           223.0
                1                                                                       

In [635]:
with pd.ExcelWriter("INVITEreport.xlsx") as writer:
    for df_name, df in report.items():
        df.to_excel(writer, sheet_name=df_name[:31])